### Calculate center x coordinate for all frames

In [ ]:
import cv2
import os
import subprocess
import numpy as np

datasets = [
    'driving_data_1.avi',
    'driving_data_2.avi',
    'driving_data_3.avi',
    'driving_data_4.avi',
    'driving_data_5.avi',
    'driving_data_6.avi',
    'driving_data_7.avi'
]

save_folder = "regression_dataset"

if os.path.exists(save_folder):
    subprocess.run(['rm', '-rf', save_folder], check=True)

# Ensure the full save path exists
save_dir = f"{save_folder}/data"
os.makedirs(save_dir, exist_ok=True)

# Create csv to save labels
csv_filename = f"{save_folder}/labels.csv"

total_frames = 0

with open(csv_filename, 'w') as labels_file:
    labels_file.write("frame_id,center_x\n")

    for data in datasets:
        video_filename = "data/" + data

        # Open the color video
        cap = cv2.VideoCapture(video_filename)

        while True:
            ret, color_img = cap.read()         # ret checks if video was read
            if not ret:
                break

            # mask the image by removing the top half
            color_img[:188, :] = 0
            color_img[:, :248] = 0
            color_img[:, 604:] = 0

            gray_image = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY) # convert to grayscale

            _, binary_mask = cv2.threshold(gray_image, 200, 255, cv2.THRESH_BINARY) # apply threshold binary filter

            # Get all white pixels (where track is)
            track_pixels = np.argwhere(binary_mask == 255)

            if len(track_pixels) > 0:
                # Calculate the exact center X-coordinate
                track_center_x = np.mean(track_pixels[:, 1])

                # Save the image first; only write the CSV entry if it succeeded
                image_filename = f"{save_dir}/frame_{total_frames}.jpg"
                if cv2.imwrite(image_filename, color_img):
                    labels_file.write(f"{total_frames},{track_center_x}\n")
                    total_frames += 1

        cap.release()

print(f"Total frames: {total_frames}")


Total frames: 15145


### Zip dataset

In [5]:
import os
!zip -r -q regression_dataset.zip regression_dataset/